In [ ]:

import pandas as pd
import os

os.makedirs('raw_data', exist_ok=True)
os.makedirs('output',   exist_ok=True)


pd.DataFrame({
    'employee_id': ['E001','E002','E003','E004','E005','E006'],
    'full_name':   ['Nguyen Van An','Tran Thi Binh','Le Van Cuong',
                    'Pham Thi Dung', None,           'Vu Thi Phuong'],
    'course':      ['Data Analyst','Data Analyst','Data Engineer',
                    'Data Analyst','Data Science','Data Analyst'],
    'enroll_date': ['2024-01-15','2024-01-16','2024-01-17',
                    '2024-01-18','2024-01-19','2024-01-20'],
    'fee_paid':    [5000000, 5000000, 6000000, None, 4500000, 5000000],
    'source':      ['Facebook','Google','Facebook','Referral','Google','Facebook']
}).to_csv('raw_data/employees.csv', index=False)


pd.DataFrame({
    'date':        ['2024-01-15','2024-01-16','2024-01-17',
                    '2024-01-18','2024-01-19','2024-01-20'],
    'platform':    ['Facebook','Facebook','Google','Google','Facebook','Google'],
    'impressions': [10000, 12000, 8000, 9500, 11000, 7500],
    'clicks':      [500, 600, 400, 475, 550, 375],
    'ad_spend':    [2000000, 2400000, 1600000, 1900000, 2200000, 1500000],
    'conversions': [3, 4, 2, 2, 3, 2]
}).to_csv('raw_data/ad_metrics.csv', index=False)

print("✅ Data created!")

✅ Data created!


In [ ]:

import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine
from utils.config import *


print(f"▶ ETL started: {datetime.now():%H:%M:%S}")


# ══════════════════════════════════
#  EXTRACT
# ══════════════════════════════════

def extract():
    print("\n[EXTRACT]")

    # df_e = pd.read_csv('raw_data/employees.csv')
    df_e = pd.read_csv('raw_data/employees_2.csv')   
    print(f"  enrollees : {len(df_e)} rows")


    # df_a = pd.read_csv("https://docs.google.com/spreadsheets/d/SHEET_ID/export?format=csv")
    # google_sheet_id = "1FQvSDuHWwOVb-qAEsz9qYMqHV8CREIw7LJVb6jJJGQQ"
    # url = 'https://docs.google.com/spreadsheets/d/' + google_sheet_id + '/export?format=csv'
    # df_a = pd.read_csv(url)   # simulate local
    
    df_a = pd.read_csv('raw_data/ad_metrics_2.csv')
    print(f"  ad_metrics: {len(df_a)} rows")

    return df_e, df_a


# ══════════════════════════════════
#  TRANSFORM
# ══════════════════════════════════

def transform(df_e, df_a):
    print("\n[TRANSFORM]")

    # ── Enrollees ──────────────────────────────────
    df_e = df_e.copy() #SettingWithCopyWarning
    df_e['full_name']    = df_e['full_name'].fillna('Unknown')
    df_e['fee_paid']     = df_e['fee_paid'].fillna(df_e['fee_paid'].median())
    df_e['enroll_date']  = pd.to_datetime(df_e['enroll_date'])
    df_e['enroll_month'] = df_e['enroll_date'].dt.month
    df_e['fee_tier']     = df_e['fee_paid'].apply(
        lambda x: 'Premium' if x > 5000000 else 'Standard'
    )
    df_e['loaded_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"  enrollees : {len(df_e)} rows transformed")

    # ── Ads ────────────────────────────────────────
    df_a = df_a.copy()
    df_a['ctr'] = round(df_a['clicks'] / df_a['impressions'] * 100, 2)
    df_a['cpc'] = round(df_a['ad_spend'] / df_a['clicks'], 0)
    df_a['cpl'] = round(df_a['ad_spend'] / df_a['conversions'], 0)
    df_a['loaded_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"  ad_metrics: {len(df_a)} rows transformed")

    return df_e, df_a


# ══════════════════════════════════
#  LOAD
# ══════════════════════════════════

def make_engine(cfg: dict):
    """Tạo kết nối SQLAlchemy từ dict config."""
    conn_str = (
        f"mysql+pymysql://{cfg['user']}:{cfg['password']}"
        f"@{cfg['host']}:{cfg['port']}/{cfg['database']}"
    )
    return create_engine(conn_str, pool_recycle=3600)


engine = make_engine(DB_CONFIG)

def load(df_e, df_a):
    print("\n[LOAD]")

    # → CSV
    df_e.to_csv('output/fact_enrollments.csv', index=False, encoding='utf-8-sig')
    df_a.to_csv('output/fact_ads.csv',         index=False, encoding='utf-8-sig')
    print("  ✅ CSV → output/")

    # → MySQL
    engine = make_engine(DB_CONFIG)
    
    # df_e.to_sql('fact_enrollments', engine, if_exists='replace', index=False)
    df_e.to_sql('fact_enrollments', engine, if_exists='append', index=False)
    # df_a.to_sql('fact_ads',         engine, if_exists='replace', index=False)
    df_a.to_sql('fact_ads',         engine, if_exists='append', index=False)
    print("  ✅ MySQL → da_analytics")


# ══════════════════════════════════
#  RUN
# ══════════════════════════════════

df_e, df_a = extract()
df_e, df_a = transform(df_e, df_a)
load(df_e, df_a)

print(f"\n✅ ETL done: {datetime.now():%H:%M:%S}")

▶ ETL started: 22:24:24

[EXTRACT]
  enrollees : 300 rows
  ad_metrics: 182 rows

[TRANSFORM]
  enrollees : 300 rows transformed
  ad_metrics: 182 rows transformed

[LOAD]
  ✅ CSV → output/
  ✅ MySQL → da_analytics

✅ ETL done: 22:24:24


In [ ]:

import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine
import logging
import os


os.makedirs('logs', exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S',
    handlers=[
        logging.FileHandler(                           # ghi ra file
            f"logs/etl_{datetime.now():%Y%m%d}.log",
            encoding='utf-8'
        ),
        logging.StreamHandler()                        # hiện terminal
    ]
)
log = logging.getLogger(__name__)


# ══════════════════════════════════
#  EXTRACT
# ══════════════════════════════════

def extract():
    log.info("[EXTRACT] start")

    df_e = pd.read_csv('raw_data/employees.csv')
    log.info(f"  enrollees  : {len(df_e)} rows")

    # df_a = pd.read_csv("https://docs.google.com/spreadsheets/d/SHEET_ID/export?format=csv")
    df_a = pd.read_csv('raw_data/ad_metrics.csv')
    log.info(f"  ad_metrics : {len(df_a)} rows")

    log.info("[EXTRACT] done")
    return df_e, df_a


# ══════════════════════════════════
#  TRANSFORM
# ══════════════════════════════════

def transform(df_e, df_a):
    log.info("[TRANSFORM] start")

    df_e = df_e.copy()
    df_e['full_name']    = df_e['full_name'].fillna('Unknown')
    df_e['fee_paid']     = df_e['fee_paid'].fillna(df_e['fee_paid'].median())
    df_e['enroll_date']  = pd.to_datetime(df_e['enroll_date'])
    df_e['enroll_month'] = df_e['enroll_date'].dt.month
    df_e['fee_tier']     = df_e['fee_paid'].apply(
        lambda x: 'Premium' if x > 5000000 else 'Standard'
    )
    df_e['loaded_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    log.info(f"  enrollees  : {len(df_e)} rows done")

    df_a = df_a.copy()
    df_a['ctr'] = round(df_a['clicks'] / df_a['impressions'] * 100, 2)
    df_a['cpc'] = round(df_a['ad_spend'] / df_a['clicks'], 0)
    df_a['cpl'] = round(df_a['ad_spend'] / df_a['conversions'], 0)
    df_a['loaded_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    log.info(f"  ad_metrics : {len(df_a)} rows done")

    log.info("[TRANSFORM] done")
    return df_e, df_a


# ══════════════════════════════════
#  LOAD
# ══════════════════════════════════
def make_engine(cfg: dict):
    """Tạo kết nối SQLAlchemy từ dict config."""
    conn_str = (
        f"mysql+pymysql://{cfg['user']}:{cfg['password']}"
        f"@{cfg['host']}:{cfg['port']}/{cfg['database']}"
    )
    return create_engine(conn_str, pool_recycle=3600)




def load(df_e, df_a):
    log.info("[LOAD] start")

    df_e.to_csv('output/fact_enrollments.csv', index=False, encoding='utf-8-sig')
    df_a.to_csv('output/fact_ads.csv',         index=False, encoding='utf-8-sig')
    log.info("  CSV saved → output/")

    engine = make_engine(DB_CONFIG)
    
    df_e.to_sql('fact_enrollments', engine, if_exists='replace', index=False)
    df_a.to_sql('fact_ads',         engine, if_exists='replace', index=False)
    log.info("  MySQL loaded → da_analytics")

    log.info("[LOAD] done")


# ══════════════════════════════════
#  RUN
# ══════════════════════════════════

log.info("=" * 40)
log.info(f"ETL started")
log.info("=" * 40)

df_e, df_a = extract()
df_e, df_a = transform(df_e, df_a)
load(df_e, df_a)

log.info("=" * 40)
log.info("ETL finished")
log.info("=" * 40)

21:17:23 | INFO | ========================================
21:17:23 | INFO | ETL started
21:17:23 | INFO | ========================================
21:17:23 | INFO | [EXTRACT] start
21:17:23 | INFO |   enrollees  : 6 rows
21:17:23 | INFO |   ad_metrics : 6 rows
21:17:23 | INFO | [EXTRACT] done
21:17:23 | INFO | [TRANSFORM] start
21:17:23 | INFO |   enrollees  : 6 rows done
21:17:23 | INFO |   ad_metrics : 6 rows done
21:17:23 | INFO | [TRANSFORM] done
21:17:23 | INFO | [LOAD] start
21:17:23 | INFO |   CSV saved → output/
21:17:23 | INFO |   MySQL loaded → da_analytics
21:17:23 | INFO | [LOAD] done
21:17:23 | INFO | ========================================
21:17:23 | INFO | ETL finished
21:17:23 | INFO | ========================================
